In [0]:
dbutils.widgets.dropdown(name='environment',defaultValue='dev',choices=['dev','qa','prod'],label='Select Environment')
env=dbutils.widgets.get('environment')
print(f'Environment is {env}')


In [0]:
bronzetable=f'saleslake_{env}.bronze_{env}.rawproduct'
print(bronzetable)
srcfileloc=f's3://saleslakekir/saleslake/src_file/DatabricksSourceFile/product.csv'
print(srcfileloc)

In [0]:
# spark.sql(f"""
# copy into {bronzetable}
# from (
#      SELECT product_id, sku, product_name, category, sub_category, brand, supplier, unit_cost, list_price, launch_date, status,
#     current_timestamp() as ingest_ts 
#     from '{srcfileloc}'
# )
# Fileformat = csv
# FORMAT_OPTIONS ("header" = "true")
# COPY_OPTIONS ("mergeSchema" = "false")

# """)


In [0]:
from pyspark.sql import functions as F
df= (spark.read.csv(srcfileloc,header=True)
     .withColumn('ingest_ts',F.current_timestamp())
)
df.write.format('delta').mode('append').option('mergeSchema','false').saveAsTable(bronzetable)
display(spark.sql(f'select * from {bronzetable}'))

In [0]:
%sql
select * from saleslake_qa.bronze_qa.rawproduct